In [14]:
import pandas as pd
import numpy as np

from sklearn.model_selection import LeaveOneOut
from sklearn.linear_model import (
    LinearRegression,
    Ridge,
    Lasso
)

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [15]:
df = pd.read_csv(
    "../data/feature_engineered_dataset.csv"
)

print(df.shape)
df.head()

(60, 25)


,CourseID,CourseName,CourseCategory,CourseType,CourseLevel,CoursePrice,CourseDuration,CourseRating,EnrollmentCount,TotalRevenue,...,TeacherRating,PriceBand,DurationBucket,RatingTier,ExperienceBucket,ExpertiseMatch,RevenuePerEnrollment,LevelEncoded,PopularityScore,RevenueTier
0,CR00001,Python Basics,Programming,Paid,Beginner,472.28,11.00,4.74,164,77453.92,...,4.97,High,Medium,High,Senior,0,472.28,1,777.36,High
1,CR00002,Java Programming,Programming,Free,Intermediate,0.00,37.70,2.43,149,0.00,...,4.97,Low,Long,Low,Senior,0,0.00,2,362.07,Low
2,CR00003,C++ for Beginners,Programming,Free,Beginner,0.00,19.53,3.85,173,0.00,...,4.58,Low,Medium,Medium,Senior,0,0.00,1,666.05,Low
3,CR00004,Advanced Python,Programming,Free,Beginner,0.00,45.13,2.88,154,0.00,...,4.58,Low,Long,Low,Senior,0,0.00,1,443.52,Low
4,CR00005,Full Stack Development,Programming,Free,Beginner,0.00,28.68,1.28,166,0.00,...,4.58,Low,Long,Low,Senior,0,0.00,1,212.48,Low


In [16]:
category_df = (
    df.groupby("CourseCategory")
      .agg(
          NumCourses=("CourseCategory", "count"),
          AvgPrice=("CoursePrice", "mean"),
          AvgDuration=("CourseDuration", "mean"),
          AvgCourseRating=("CourseRating", "mean"),
          AvgExperience=("YearsOfExperience", "mean"),
          TotalEnrollments=("EnrollmentCount", "sum"),
          CategoryRevenue=("TotalRevenue", "sum")
      )
      .reset_index()
)

category_df

,CourseCategory,NumCourses,AvgPrice,AvgDuration,AvgCourseRating,AvgExperience,TotalEnrollments,CategoryRevenue
0,Artificial Intelligence,5,248.820,17.906,3.036,22.2,829,202750.67
1,Business,5,219.798,24.046,2.688,22.8,833,181527.58
2,Cybersecurity,5,78.852,36.308,2.902,24.0,819,59139.00
3,Data Science,5,83.450,27.992,3.316,22.2,916,77052.32
4,Design,5,55.274,32.774,3.162,22.2,827,43113.72
5,Digital Marketing,5,77.946,17.796,3.656,22.8,808,56261.32
6,Finance,5,34.580,28.690,3.010,23.4,864,28106.20
7,Machine Learning,5,0.156,31.356,2.658,21.0,819,133.38
8,Marketing,5,0.000,30.348,3.722,22.2,806,0.00
9,Programming,5,94.456,28.408,3.036,22.8,806,77453.92


In [22]:
category_df["RevenuePerCourse"] = (
    category_df["CategoryRevenue"]
    / category_df["NumCourses"]
)

category_df["EnrollmentsPerCourse"] = (
    category_df["TotalEnrollments"]
    / category_df["NumCourses"]
)

category_df.head()

,CourseCategory,NumCourses,AvgPrice,AvgDuration,AvgCourseRating,AvgExperience,TotalEnrollments,CategoryRevenue,RevenuePerCourse,EnrollmentsPerCourse
0,Artificial Intelligence,5,248.820,17.906,3.036,22.2,829,202750.67,40550.134,165.8
1,Business,5,219.798,24.046,2.688,22.8,833,181527.58,36305.516,166.6
2,Cybersecurity,5,78.852,36.308,2.902,24.0,819,59139.00,11827.800,163.8
3,Data Science,5,83.450,27.992,3.316,22.2,916,77052.32,15410.464,183.2
4,Design,5,55.274,32.774,3.162,22.2,827,43113.72,8622.744,165.4


In [23]:
X = category_df.drop(
    columns=[
        "CategoryRevenue",
        "CourseCategory",
        "RevenuePerCourse",
        "EnrollmentsPerCourse"
    ]
)

y = category_df["CategoryRevenue"]

print(X.columns.tolist())

['NumCourses', 'AvgPrice', 'AvgDuration', 'AvgCourseRating', 'AvgExperience', 'TotalEnrollments']


In [21]:
print(category_df.columns.tolist())

['CourseCategory', 'NumCourses', 'AvgPrice', 'AvgDuration', 'AvgCourseRating', 'AvgExperience', 'TotalEnrollments', 'CategoryRevenue', 'RevenuePerCourse', 'EnrollmentsPerCourse']


In [8]:
models = {
    "Linear": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "Lasso": Lasso(alpha=1.0)
}

In [24]:
category_df[[
    "AvgPrice",
    "TotalEnrollments",
    "CategoryRevenue"
]].corr()

,AvgPrice,TotalEnrollments,CategoryRevenue
AvgPrice,1.000000,0.003091,0.998404
TotalEnrollments,0.003091,1.000000,0.044620
CategoryRevenue,0.998404,0.044620,1.000000


In [9]:
loo = LeaveOneOut()

In [10]:
def evaluate_model(model, X, y):

    actuals = []
    predictions = []

    for train_idx, test_idx in loo.split(X):

        X_train = X.iloc[train_idx]
        X_test = X.iloc[test_idx]

        y_train = y.iloc[train_idx]
        y_test = y.iloc[test_idx]

        model.fit(X_train, y_train)

        pred = model.predict(X_test)

        actuals.append(y_test.values[0])
        predictions.append(pred[0])

    mae = mean_absolute_error(
        actuals,
        predictions
    )

    rmse = np.sqrt(
        mean_squared_error(
            actuals,
            predictions
        )
    )

    r2 = r2_score(
        actuals,
        predictions
    )

    return mae, rmse, r2

In [11]:
results = []

for name, model in models.items():

    mae, rmse, r2 = evaluate_model(
        model,
        X,
        y
    )

    results.append({
        "Model": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })

results_df = pd.DataFrame(results)

results_df.sort_values(
    "R2",
    ascending=False
)

,Model,MAE,RMSE,R2
0,Linear,5.547452e-11,6.922260e-11,1.000000
1,Ridge,1.798873e-03,2.379617e-03,1.000000
2,Lasso,1.511637e+03,1.953279e+03,0.999167


In [12]:
from joblib import dump

best_model = Ridge(alpha=1.0)

best_model.fit(X, y)

dump(
    best_model,
    "../models/category_revenue_model.pkl"
)

print("Category model saved")

Category model saved
